In [ ]:
%pip install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer,util


In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
question_pool = [
    "Hello",
    "What options are available on the menu?",
    "Do you offer burgers?",
    "Can you tell me about the pizzas?",
    "What types of pasta do you have?",
    "Do you serve biryani?",
    "Do you have BBQ dishes?",
    "Is dessert available?",
    "What drinks are served?",
    "Can I reserve a table?",
    "Thank you",
    "Goodbye"
]


answer_pool = [
    "Hello! Welcome to Food Lounge. What would you like to have today?",
    "Our menu is full of delicious options. We offer pizzas, burgers, pasta, biryani, BBQ, desserts, and much more.",
    "We serve juicy and tasty burgers. Options include Classic Beef, Chicken Supreme, and Veggie Delight.",
    "We have freshly baked pizzas. Options include Margherita, Pepperoni, BBQ Chicken, and Veggie Special.",
    "Our pasta menu features creamy Alfredo, spicy Arrabiata, and classic Bolognese.",
    "We offer flavorful biryani. Choices include Chicken Biryani, Mutton Biryani, and Veg Biryani.",
    "Our BBQ dishes are tender and smoky. Options include BBQ Wings, Ribs, and BBQ Platters.",
    "We have a variety of desserts. Options include Chocolate Lava Cake, Cheesecake, and Ice Cream Sundae.",
    "We offer a range of drinks. Options include Lemonade, Mojito, Cold Coffee, and Fresh Juices.",
    "Of course. To book a table, please call us at +123-456-7890 or visit our website to reserve online.",
    "You're welcome. Enjoy your meal.",
    "Goodbye. We look forward to seeing you again."
]



In [ ]:
question_embeddings = model.encode(question_pool, convert_to_tensor=True)


In [ ]:
import numpy as np
     


In [ ]:
question_embeddings = np.array(question_embeddings).astype('float32')
     


In [ ]:

np.save('restaurant_embeddings.npy',question_embeddings)
     


In [ ]:

question_embeddings = np.load('restaurant_embeddings.npy')
     


In [ ]:

%pip install faiss-cpu

In [ ]:
import faiss


In [ ]:
dimension = question_embeddings.shape[1]  # usually 384
index = faiss.IndexFlatL2(dimension)
index.add(question_embeddings)

In [ ]:
SIMILARITY_THRESHOLD = 0.4


In [ ]:
def chatbot_reply(user_input):
    user_embedding = model.encode(user_input, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(user_embedding, question_embeddings)

    best_score = cosine_scores.max().item()
    best_index = cosine_scores.argmax().item()

    if best_score >= SIMILARITY_THRESHOLD:
        return answer_pool[best_index]
    else:
        return "I'm not sure about that. Try asking about food, drinks, or bookings!"

while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit', 'bye']:
        print("Bot: Goodbye! 👋")
        break
    response = chatbot_reply(user_input)
    print("Bot:", response)